# Proposta Marcelo - Leitura de dados

Este notebook inicia o fluxo do projeto. O objetivo desta etapa é configurar a origem dos dados, listar as bases disponíveis e confirmar que conseguimos ler os arquivos Excel necessários para o relatório estadual.

## 1. Importar bibliotecas

Usamos bibliotecas padrão do Python para caminhos, datas e ambiente, além de `pandas` para ler as planilhas Excel.

In [9]:
from datetime import datetime
from pathlib import Path
import os

import pandas as pd


## 2. Localizar a raiz do projeto

Esta célula procura automaticamente a pasta raiz do repositório. Assim, o notebook deve funcionar tanto quando o VS Code abrir o projeto pela raiz quanto quando o kernel iniciar dentro da pasta `notebooks`.

In [10]:
def encontrar_raiz_projeto(inicio: Path | None = None) -> Path:
    """Procura a raiz do projeto a partir do diretório atual."""
    inicio = (inicio or Path.cwd()).resolve()

    for candidato in [inicio, *inicio.parents]:
        tem_notebooks = (candidato / "notebooks").exists()
        tem_requirements = (candidato / "requirements.txt").exists()
        if tem_notebooks and tem_requirements:
            return candidato

    raise FileNotFoundError(
        "Não encontrei a raiz do projeto. Abra o VS Code na pasta relatorio_dados_damei "
        "ou execute o notebook a partir de uma subpasta do repositório."
    )


PROJECT_ROOT = encontrar_raiz_projeto()
print(f"Raiz do projeto: {PROJECT_ROOT}")


Raiz do projeto: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei


## 3. Configurar origem dos dados e pasta de saída

O GitHub guarda código, documentação e template. O Google Drive é o repositório operacional dos dados e dos relatórios gerados.

Existem apenas dois modos:

- `local`: roda no VS Code e usa uma cópia local/mock em `dados_brutos/dado_atual`.
- `google_drive`: roda no Google Colab, monta o Google Drive e lê os dados oficiais do Drive.

In [11]:
# Opções: "local" ou "google_drive".
# Use "local" no VS Code.
# Use "google_drive" no Google Colab.
MODO_DADOS = "local"

# UF usada nas próximas etapas do projeto.
UF_INTERESSE = "MG"

# Caminhos no Google Drive para execução no Colab.
GOOGLE_DRIVE_RAW_DIR = Path("/content/drive/MyDrive/MDA/DAMEI_Relatorio_Dados/dados_brutos/dado_atual")
GOOGLE_DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/MDA/dado_atual")

# Subpastas usadas somente no modo local.
RAW_SUBDIR = Path("dados_brutos") / "dado_atual"
OUTPUT_SUBDIR = Path("relatorios_gerados")

# Data de execução usada futuramente no nome dos arquivos gerados.
RUN_TIMESTAMP = datetime.now()
RUN_YYYYMM = RUN_TIMESTAMP.strftime("%Y%m")
RUN_DATETIME = RUN_TIMESTAMP.strftime("%Y%m%d%H%M%S")


## 4. Resolver caminhos

Esta célula transforma a configuração acima em caminhos finais de leitura e gravação.

No modo `local`, o notebook usa as pastas do próprio repositório. No modo `google_drive`, o notebook exige Google Colab, monta o Drive e usa os caminhos `/content/drive/...`.

In [12]:
def esta_no_colab() -> bool:
    """Retorna True quando o notebook está rodando no Google Colab."""
    return "COLAB_RELEASE_TAG" in os.environ

if MODO_DADOS == "local":
    DATA_PROJECT_ROOT = PROJECT_ROOT
    RAW_DIR = PROJECT_ROOT / RAW_SUBDIR
    OUTPUT_BASE_DIR = PROJECT_ROOT / OUTPUT_SUBDIR
    OUTPUT_DIR = OUTPUT_BASE_DIR / RUN_YYYYMM
elif MODO_DADOS == "google_drive":
    if not esta_no_colab():
        raise RuntimeError(
            'MODO_DADOS = "google_drive" só deve ser usado no Google Colab. '
            'No VS Code local, use MODO_DADOS = "local".'
        )

    from google.colab import drive

    drive.mount("/content/drive")
    DATA_PROJECT_ROOT = GOOGLE_DRIVE_RAW_DIR.parents[1]
    RAW_DIR = GOOGLE_DRIVE_RAW_DIR
    OUTPUT_DIR = GOOGLE_DRIVE_OUTPUT_DIR
else:
    raise ValueError('MODO_DADOS deve ser "local" ou "google_drive".')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Modo de dados:     {MODO_DADOS}")
print(f"UF de interesse:   {UF_INTERESSE}")
print(f"Projeto de dados:  {DATA_PROJECT_ROOT}")
print(f"Pasta de dados:    {RAW_DIR}")
print(f"Pasta de saída:    {OUTPUT_DIR}")


Modo de dados:     local
UF de interesse:   MG
Projeto de dados:  C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei
Pasta de dados:    C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_atual
Pasta de saída:    C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\relatorios_gerados\202604


## 5. Validar pastas de dados e saída

Antes de ler qualquer arquivo, validamos se a pasta de dados existe. A pasta de saída é criada automaticamente quando ainda não existir. No modo `google_drive`, ela usa o caminho configurado em `GOOGLE_DRIVE_OUTPUT_DIR`.

In [13]:
if not RAW_DIR.exists():
    raise FileNotFoundError(f"A pasta de dados não existe: {RAW_DIR}")

print("Pasta de dados encontrada.")
print("Pasta de saída pronta.")


Pasta de dados encontrada.
Pasta de saída pronta.


## 6. Listar arquivos Excel

Aqui verificamos quais arquivos `.xlsx` estão disponíveis na pasta de dados configurada.

In [14]:
excel_files = sorted(RAW_DIR.glob("*.xlsx"))

if not excel_files:
    raise FileNotFoundError(f"Nenhum arquivo .xlsx encontrado em {RAW_DIR}")

print(f"Foram encontrados {len(excel_files)} arquivos Excel:")
for arquivo in excel_files:
    tamanho_mb = arquivo.stat().st_size / (1024 * 1024)
    print(f"- {arquivo.name} ({tamanho_mb:.2f} MB)")


Foram encontrados 7 arquivos Excel:
- ater_ate_2026_03_gerado_em_20260410151127.xlsx (0.03 MB)
- caf_dap_ativos_ate_2026_03_gerado_em_20260410152650.xlsx (0.31 MB)
- GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx (0.09 MB)
- mais_alimentos_gaia_202604151554.xlsx (0.22 MB)
- pncf_2026_03_gerado_em_20260410170006.xlsx (0.02 MB)
- PNRA_2026_2026_04_15.xlsx (0.29 MB)
- pronaf_gaia_20260414.xlsx (0.64 MB)


## 7. Identificar arquivos por política

Nesta etapa fazemos um primeiro mapeamento automático entre nomes de arquivos e políticas públicas. Essa lógica ainda é simples, mas já ajuda a validar se temos todas as bases esperadas.

In [15]:
arquivos_encontrados = {
    "caf_dap": None,
    "ater": None,
    "pronaf": None,
    "mais_alimentos": None,
    "pncf": None,
    "garantia_safra": None,
    "pnra": None,
}

for arquivo in excel_files:
    nome = arquivo.name.lower()

    if "caf" in nome:
        arquivos_encontrados["caf_dap"] = arquivo
    elif "ater" in nome:
        arquivos_encontrados["ater"] = arquivo
    elif "pronaf" in nome:
        arquivos_encontrados["pronaf"] = arquivo
    elif "mais_alimentos" in nome or "mais-alimentos" in nome:
        arquivos_encontrados["mais_alimentos"] = arquivo
    elif "pncf" in nome:
        arquivos_encontrados["pncf"] = arquivo
    elif "garantia-safra" in nome or "garantia_safra" in nome:
        arquivos_encontrados["garantia_safra"] = arquivo
    elif "pnra" in nome:
        arquivos_encontrados["pnra"] = arquivo

resumo_arquivos = pd.DataFrame(
    [
        {
            "politica": politica,
            "arquivo": arquivo.name if arquivo else "NÃO ENCONTRADO",
            "encontrado": arquivo is not None,
        }
        for politica, arquivo in arquivos_encontrados.items()
    ]
)

resumo_arquivos


,politica,arquivo,encontrado
0,caf_dap,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,True
1,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,True
2,pronaf,pronaf_gaia_20260414.xlsx,True
3,mais_alimentos,mais_alimentos_gaia_202604151554.xlsx,True
4,pncf,pncf_2026_03_gerado_em_20260410170006.xlsx,True
5,garantia_safra,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx,True
6,pnra,PNRA_2026_2026_04_15.xlsx,True


## 8. Inspecionar abas e colunas

Agora abrimos cada arquivo com `pandas.ExcelFile`, listamos as abas e coletamos os nomes das colunas. Este teste ainda não transforma os dados; ele apenas confirma que os arquivos são legíveis.

In [16]:
linhas = []

for politica, arquivo in arquivos_encontrados.items():
    if arquivo is None:
        linhas.append(
            {
                "politica": politica,
                "arquivo": "NÃO ENCONTRADO",
                "aba": None,
                "qtd_colunas": None,
                "colunas": None,
            }
        )
        continue

    xls = pd.ExcelFile(arquivo)
    for aba in xls.sheet_names:
        colunas = pd.read_excel(arquivo, sheet_name=aba, nrows=0).columns.astype(str).tolist()
        linhas.append(
            {
                "politica": politica,
                "arquivo": arquivo.name,
                "aba": aba,
                "qtd_colunas": len(colunas),
                "colunas": ", ".join(colunas[:12]) + (" ..." if len(colunas) > 12 else ""),
            }
        )

df_estrutura = pd.DataFrame(linhas)
df_estrutura


,politica,arquivo,aba,qtd_colunas,colunas
0,caf_dap,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,Orientações,1,Orientações
1,caf_dap,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,DADOS,9,"dt_referencia, dt_geracao, cod_ibge, nome_muni..."
2,caf_dap,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,TOTALIZADORES,1,Planilha de Dados - Totalizadores
3,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,Orientações,1,Orientações
4,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,DADOS,7,"dt_referencia, dt_geracao, cod_ibge, nome_muni..."
5,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,TOTALIZADORES,1,Planilha de Dados - Totalizadores
6,pronaf,pronaf_gaia_20260414.xlsx,Resumo,1,Orientacoes gerais sobre a planilha de dados:
7,pronaf,pronaf_gaia_20260414.xlsx,Totalizadores,17,"dt_referencia, dt_geracao, uf, cod_ibge_uf, AN..."
8,pronaf,pronaf_gaia_20260414.xlsx,Dados,19,"dt_referencia, dt_geracao, ANO, nome_municipio..."
9,mais_alimentos,mais_alimentos_gaia_202604151554.xlsx,Resumo,1,Orientações gerais sobre a planilha de dados:


## 9. Testar leitura das abas principais

Por fim, carregamos as abas principais de cada política e mostramos o tamanho de cada tabela. Se esta célula rodar sem erro, o teste de leitura está aprovado.

In [17]:
abas_principais = {
    "caf_dap": "DADOS",
    "ater": "DADOS",
    "pronaf": "Dados",
    "mais_alimentos": "Dados",
    "pncf": "DADOS",
    "garantia_safra": "DADOS",
    "pnra": "DADOS",
}

tabelas = {}
resumo_leitura = []

for politica, aba in abas_principais.items():
    arquivo = arquivos_encontrados.get(politica)

    if arquivo is None:
        resumo_leitura.append(
            {
                "politica": politica,
                "status": "arquivo não encontrado",
                "linhas": None,
                "colunas": None,
            }
        )
        continue

    df = pd.read_excel(arquivo, sheet_name=aba)
    tabelas[politica] = df
    resumo_leitura.append(
        {
            "politica": politica,
            "status": "ok",
            "linhas": df.shape[0],
            "colunas": df.shape[1],
        }
    )

pd.DataFrame(resumo_leitura)


,politica,status,linhas,colunas
0,caf_dap,ok,5526,9
1,ater,ok,328,7
2,pronaf,ok,4796,19
3,mais_alimentos,ok,3935,8
4,pncf,ok,77,9
5,garantia_safra,ok,1215,9
6,pnra,ok,5600,10
